# Volume arterial de referência do ImageCAS

Análise do volume físico das máscaras coronarianas (`label == 1`), na resolução nativa. Relaciona a distribuição dos volumes aos resultados canônicos Mid/High, sem executar o pipeline.

O cálculo utiliza o espaçamento de cada arquivo: $V_{\mathrm{mL}} = N_{\mathrm{voxels}}\,s_xs_ys_z/1000$, com espaçamentos em mm. Assim, 1 mL equivale a 1.000 mm³. O valor representa o espaço ocupado pela anotação, não fluxo sanguíneo.


## 1. Configuração

O CSV é salvo após cada exame. Nas próximas execuções, somente arquivos novos ou com erro são calculados. Use `FORCE_RECOMPUTE = True` se as máscaras concluídas forem alteradas. O diretório do dataset é resolvido por `IMAGECAS_BASE_PATH` ou pelos caminhos conhecidos do projeto.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# O bootstrap precede os imports compartilhados de utils.
REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src").is_dir()
)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from utils.comparison_utils.artery_volume import (  # noqa: E402
    compare_artery_volume_cohorts,
    prepare_artery_volume_dice_data,
    prepare_predicted_artery_volumes,
    summarize_artery_volume_bands,
)
from utils.comparison_utils.io import load_split_results  # noqa: E402
from utils.project.analysis.artery_volume import (  # noqa: E402
    classify_artery_volume_outliers,
    load_or_compute_artery_volumes,
)
from utils.project.runtime.notebook_env import (  # noqa: E402
    configure_notebook_environment,
    get_default_split_paths,
)

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
CSV_PATH = (
    REPO_ROOT
    / "output"
    / "segmentation"
    / "analysis"
    / "imagecas_artery_volume"
    / "artery_volumes.csv"
)
FORCE_RECOMPUTE = False
PLOT_SPLIT = "test"  # train, val, test ou all (30 + 270 + 700 exames).
HISTOGRAM_BINS = 30
EXTREME_CASE_COUNT = 5
OUTLIER_METHOD = "std"  # Alternativas: "iqr" e "std".
IQR_MULTIPLIER = 1.5
STD_MULTIPLIER = 2.0  # 2.0

In [ ]:
# Rótulos e gráficos Plotly ficam no submódulo `view`; o pareamento fica em comparison_utils.
from eda.images.view.artery_volume_views import (
    SPLIT_LABELS,
    plot_volume_band_performance,
    plot_volume_dice_delta,
    plot_volume_dice_panels,
    plot_volume_histogram,
)

## 2. Carregamento ou cálculo

O helper lê somente as máscaras, uma por vez, e converte as unidades espaciais NIfTI para mm (cabeçalhos sem unidade seguem a convenção mm do ImageCAS). Erros ficam registrados no CSV sem receber volume zero. Sem o dataset montado, o CSV existente continua consultável, com aviso de que novos arquivos não puderam ser verificados.


In [ ]:
# Retoma o CSV sem reler as máscaras já concluídas.
results_df = load_or_compute_artery_volumes(
    CSV_PATH,
    force_recompute=FORCE_RECOMPUTE,
)
valid_df = results_df.loc[results_df["status"].eq("ok")].copy()
error_df = results_df.loc[results_df["status"].eq("error")].copy()
print(f"CSV: {CSV_PATH.relative_to(REPO_ROOT)}")

In [ ]:
display(
    pd.DataFrame(
        {
            "Situação": ["Exames registrados", "Concluídos", "Com erro"],
            "Quantidade": [len(results_df), len(valid_df), len(error_df)],
        }
    )
)

## 3. Estatísticas do volume em mL

Somente exames com status `ok` entram nas estatísticas. O desvio-padrão é amostral; 25%, 50% e 75% correspondem ao primeiro quartil, mediana e terceiro quartil. Um volume realmente vazio permanece zero; uma falha de leitura permanece ausente.


In [ ]:
if valid_df.empty:
    print("Não há exames concluídos para calcular estatísticas.")
else:
    statistics = valid_df["volume_ml"].describe().drop("count")
    statistics.index = [
        "Média",
        "Desvio-padrão",
        "Mínimo",
        "Primeiro quartil",
        "Mediana",
        "Terceiro quartil",
        "Máximo",
    ]
    display(statistics.rename("Volume (mL)").to_frame().round(4))

## 4. Volumes baixos, altos e outliers

Por padrão, os limites são $Q_1-1{,}5\,IQR$ e $Q_3+1{,}5\,IQR$, com $IQR=Q_3-Q_1$. Valores exatamente nos limites não são outliers. A alternativa `OUTLIER_METHOD = "std"` usa média ± `STD_MULTIPLIER` desvios-padrão amostrais.

Os limites são descritivos, calculados sobre todos os volumes válidos do CSV. A distância padronizada é $z=(V-\bar V)/s$: valores negativos indicam volumes abaixo da média; positivos, acima. Um outlier volumétrico não implica falha ou anotação incorreta.


In [ ]:
# A classificação é feita em memória; o CSV de volumes não é modificado.
classified_volumes_df, outlier_limits = classify_artery_volume_outliers(
    results_df,
    method=OUTLIER_METHOD,
    iqr_multiplier=IQR_MULTIPLIER,
    std_multiplier=STD_MULTIPLIER,
)
display(
    pd.DataFrame(
        {
            "Critério": [OUTLIER_METHOD.upper()],
            "Exames válidos": [outlier_limits["exam_count"]],
            "Limite inferior (mL)": [outlier_limits["lower_limit_ml"]],
            "Limite superior (mL)": [outlier_limits["upper_limit_ml"]],
        }
    ).round(4)
)

In [ ]:
counts = classified_volumes_df["volume_outlier_group"].value_counts()
outlier_counts_df = pd.DataFrame(
    {
        "Grupo": ["Volume baixo (outlier)", "Dentro da faixa", "Volume alto (outlier)"],
        "Quantidade": [int(counts.get(key, 0)) for key in ("low", "in_range", "high")],
    }
)
outlier_counts_df["Percentual (%)"] = (
    outlier_counts_df["Quantidade"] / len(classified_volumes_df) * 100
    if len(classified_volumes_df)
    else float("nan")
)
display(outlier_counts_df.round(2))

In [ ]:
outlier_exams_df = classified_volumes_df.loc[
    classified_volumes_df["is_volume_outlier"],
    ["IMG_ID", "volume_ml", "volume_outlier_group", "volume_z_score"],
].sort_values("volume_ml")
if outlier_exams_df.empty:
    print("Nenhum outlier identificado pelo critério selecionado.")
else:
    display(
        outlier_exams_df.replace(
            {"volume_outlier_group": {"low": "Baixo", "high": "Alto"}}
        )
        .rename(
            columns={
                "volume_ml": "Volume (mL)",
                "volume_outlier_group": "Grupo",
                "volume_z_score": "Distância da média (DP)",
            }
        )
        .reset_index(drop=True)
        .round(4)
    )

## 5. Distribuição dos volumes

Cada observação corresponde a um exame. A curva suavizada ajuda a visualizar a distribuição; as linhas verticais indicam média e mediana.


In [ ]:
volume_histogram_figure = plot_volume_histogram(
    valid_df, outlier_limits, HISTOGRAM_BINS
)
if volume_histogram_figure is not None:
    volume_histogram_figure.show()

## 6. Exames com menores e maiores volumes

Os extremos são candidatos à inspeção, não erros automáticos: diferenças anatômicas e de extensão da anotação também alteram o volume.


In [ ]:
if valid_df.empty:
    print("Não há exames concluídos para selecionar os menores volumes.")
else:
    display(
        valid_df.nsmallest(EXTREME_CASE_COUNT, "volume_ml")
        .loc[:, ["IMG_ID", "volume_ml", "voxel_count", "label_file"]]
        .rename(columns={"volume_ml": "Volume (mL)", "voxel_count": "Voxels"})
        .reset_index(drop=True)
    )

In [ ]:
if valid_df.empty:
    print("Não há exames concluídos para selecionar os maiores volumes.")
else:
    display(
        valid_df.nlargest(EXTREME_CASE_COUNT, "volume_ml")
        .loc[:, ["IMG_ID", "volume_ml", "voxel_count", "label_file"]]
        .rename(columns={"volume_ml": "Volume (mL)", "voxel_count": "Voxels"})
        .reset_index(drop=True)
    )

## 7. Canonical Mid/High com e sem os outliers

A comparação usa somente as referências oficiais de `canonical`, por treino, validação, teste e total disponível. Os mesmos IDs classificados como outliers são excluídos em ambas as resoluções. Exames sem volume conhecido são informados e preservados; falhas documentadas sem Dice recebem zero.

Esta é uma análise descritiva de sensibilidade à composição da coorte. A exclusão usa o volume do ground truth e não representa melhoria validada do algoritmo nem substitui o resultado global. Nenhum resultado canônico é alterado.


In [ ]:
# Carrega resultados salvos: não executa segmentações nem procura runs alternativos.
canonical_frames = {}
canonical_configs = {}
canonical_sources = []
comparison_df = pd.DataFrame()
if classified_volumes_df.empty:
    print("Não há volumes válidos para analisar a exclusão de outliers.")
else:
    split_paths = get_default_split_paths(REPO_ROOT)
    for resolution in ("mid_res", "high_res"):
        for split in ("train", "val", "test"):
            source = split_paths.get(resolution, {}).get(split)
            message = "Disponível"
            try:
                frame = load_split_results(split_paths, resolution, split)
            except FileNotFoundError as error:
                frame = None
                message = str(error)
            if frame is not None:
                canonical_frames[resolution, split] = frame
                config_path = (
                    source.parent if source.name == "numeric" else source
                ) / "config/effective_pipeline_config.json"
                if config_path.is_file():
                    with config_path.open() as handle:
                        canonical_configs[resolution, split] = json.load(handle)
                else:
                    message = "Resultados disponíveis; configuração ausente para volume predito"
            elif message == "Disponível":
                message = "Canonical ausente"
            canonical_sources.append(
                {
                    "Resolução": "Mid" if resolution == "mid_res" else "High",
                    "Conjunto": split,
                    "Exames": len(frame) if frame is not None else 0,
                    "Origem": str(source.relative_to(REPO_ROOT)) if source else "—",
                    "Situação": message,
                }
            )
    if canonical_frames:
        comparison_df = compare_artery_volume_cohorts(
            canonical_frames, classified_volumes_df
        )
        comparison_df["Resolução"] = comparison_df["resolution"].map(
            {"mid_res": "Mid", "high_res": "High"}
        )
        comparison_df["Conjunto"] = comparison_df["split"].map(
            {
                "train": "Treino",
                "val": "Validação",
                "test": "Teste",
                "all": "Todos disponíveis",
            }
        )
display(pd.DataFrame(canonical_sources))

In [ ]:
if comparison_df.empty:
    print("Não há referências canônicas disponíveis para comparar.")
else:
    display(
        comparison_df[
            [
                "Resolução",
                "Conjunto",
                "total_exams",
                "excluded_exams",
                "remaining_exams",
                "dice_before",
                "dice_after",
                "delta_dice",
            ]
        ]
        .rename(
            columns={
                "total_exams": "N original",
                "excluded_exams": "Outliers excluídos",
                "remaining_exams": "N sem outliers",
                "dice_before": "Dice original",
                "dice_after": "Dice sem outliers",
                "delta_dice": "Δ Dice",
            }
        )
        .round(4)
    )

In [ ]:
if not comparison_df.empty:
    display(
        comparison_df[
            [
                "Resolução",
                "Conjunto",
                "ostia_success_count_before",
                "ostia_success_count_after",
                "ostia_success_percent_before",
                "ostia_success_percent_after",
                "delta_ostia_success_pp",
            ]
        ]
        .rename(
            columns={
                "ostia_success_count_before": "Sucessos originais",
                "ostia_success_count_after": "Sucessos sem outliers",
                "ostia_success_percent_before": "Sucesso original (%)",
                "ostia_success_percent_after": "Sucesso sem outliers (%)",
                "delta_ostia_success_pp": "Δ sucesso (p.p.)",
            }
        )
        .round(2)
    )

In [ ]:
if not comparison_df.empty:
    missing_volume_df = comparison_df.loc[
        comparison_df["missing_volume_count"].gt(0),
        ["Resolução", "Conjunto", "missing_volume_count"],
    ]
    if missing_volume_df.empty:
        print("Todos os exames canônicos têm volume disponível no CSV.")
    else:
        print("Exames sem volume foram preservados, não excluídos automaticamente.")
        display(
            missing_volume_df.rename(columns={"missing_volume_count": "Sem volume"})
        )

## 8. Volume arterial e desempenho canônico

Selecione `PLOT_SPLIT` na configuração (padrão: teste). Use `all` para reunir treino, validação e teste, totalizando 1.000 exames quando as três referências estiverem disponíveis. A tabela de cobertura identifica conjuntos ausentes; IDs duplicados entre conjuntos são rejeitados. O volume predito usa a configuração efetiva de cada origem, mesmo na seleção completa. O eixo de volume usa a máscara coronariana de referência em mL. Os outliers continuam incluídos; a análise é descritiva, não um ajuste do pipeline.

A tabela informa IDs sem volume ou sem resultado na outra resolução. A figura interativa alterna entre todos os exames com volume conhecido em cada resolução e os IDs comuns com ambos os óstios corretos ou toleráveis nas duas resoluções. O delta é calculado nos IDs comuns com volume, incluindo falhas documentadas.


In [ ]:
volume_dice_df = pd.DataFrame()
volume_dice_pairs = pd.DataFrame()
if classified_volumes_df.empty:
    print("Sem volumes válidos para os gráficos.")
else:
    volume_dice_df, volume_dice_pairs, volume_dice_coverage = (
        prepare_artery_volume_dice_data(
            canonical_frames,
            classified_volumes_df,
            split=PLOT_SPLIT,
        )
    )
    display(
        volume_dice_coverage.rename(
            columns={
                "resolution": "Resolução",
                "split": "Conjunto",
                "source_available": "Canonical disponível",
                "missing_splits": "Conjuntos ausentes",
                "total_exams": "N canônico",
                "plotted_exams": "N com volume",
                "missing_volume_count": "Sem volume",
                "missing_volume_ids": "IDs sem volume",
                "missing_pair_count": "Sem par",
                "missing_pair_ids": "IDs sem par",
            }
        )
    )
volume_plot_xlim = (
    (0, max(float(volume_dice_df["volume_ml"].max()) * 1.05, 1.0))
    if not volume_dice_df.empty
    else (0, 1)
)

### 8.1 Volume × Dice: comparação interativa

O menu alterna entre todos os exames e a coorte comum com óstios válidos nas duas resoluções, mantendo os limites dos eixos. As cores distinguem sucesso e falha dos óstios; círculos e X distinguem volumes dentro da faixa e outliers. Passe o mouse para consultar ID, resolução, volume, Dice e classificações.

As falhas documentadas sem Dice são representadas por zero. O filtro condicionado não altera os resultados gerais nem exclui outliers automaticamente.


In [ ]:
volume_dice_figure = plot_volume_dice_panels(
    volume_dice_df,
    volume_dice_pairs,
    f"{SPLIT_LABELS[PLOT_SPLIT]}: volume arterial × Dice",
    volume_plot_xlim=volume_plot_xlim,
)
if volume_dice_figure is not None:
    volume_dice_figure.show()

### 8.2 Volume × diferença de Dice entre resoluções

Cada ponto é um exame pareado. Valores positivos favorecem High; negativos favorecem Mid. A cor distingue sucesso dos óstios nas duas resoluções de falha em pelo menos uma delas. Os outliers são mantidos.


In [ ]:
volume_delta_figure = plot_volume_dice_delta(
    volume_dice_pairs,
    split_name=PLOT_SPLIT,
    volume_plot_xlim=volume_plot_xlim,
)
if volume_delta_figure is not None:
    volume_delta_figure.show()

### 8.3 Desempenho por faixa de volume

Os quartis são calculados com um volume por exame do conjunto selecionado; os limites são os mesmos em Mid e High. Limites repetidos reduzem a quantidade de faixas. Todos os exames com referência conhecida permanecem incluídos, inclusive falhas e outliers. As barras de erro mostram DP amostral, não intervalo de confiança; para N=1, o DP é indisponível.


In [ ]:
volume_band_summary, volume_band_bounds = summarize_artery_volume_bands(volume_dice_df)
if not volume_band_bounds.empty:
    print(f"Faixas efetivas: {len(volume_band_bounds)} (até quatro quartis).")
    display(
        volume_band_bounds.rename(
            columns={
                "band": "Faixa",
                "lower_ml": "Limite inferior (mL)",
                "upper_ml": "Limite superior (mL)",
            }
        ).round(3)
    )

In [ ]:
volume_band_figure = plot_volume_band_performance(
    volume_band_summary,
    split_name=PLOT_SPLIT,
)
if volume_band_figure is not None:
    volume_band_figure.show()

### 8.4 Volume predito × volume de referência

O volume predito usa os voxels após a morfologia, multiplicados pelo volume físico do voxel na grade processada. Os espaçamentos nativos vêm do CSV de volumes e os fatores de redução vêm da configuração efetiva da mesma fonte canônica. Configurações, contagens ou geometrias ausentes não são substituídas por valores presumidos.

Na linha y=x os volumes são iguais; abaixo dela, o volume predito é menor; acima, maior. Isso não garante correspondência espacial nem diagnostica sozinho vazamento. Predições registradas com zero voxels são preservadas; valores ausentes são reportados e excluídos apenas deste gráfico. O menu condicionado exige dados exibíveis e óstios válidos nas duas resoluções.


In [ ]:
predicted_volume_df = pd.DataFrame()
if not classified_volumes_df.empty:
    predicted_volume_df, predicted_volume_coverage = prepare_predicted_artery_volumes(
        canonical_frames,
        classified_volumes_df,
        canonical_configs,
        split=PLOT_SPLIT,
    )
    display(
        predicted_volume_coverage.rename(
            columns={
                "resolution": "Resolução",
                "total_exams": "N canônico",
                "missing_reference_count": "Sem volume de referência",
                "valid_predictions": "Volumes preditos válidos",
                "excluded_ids": "IDs excluídos",
                "status_counts": "Situação do cálculo",
            }
        )
    )

In [ ]:
predicted_volume_figure = None
if not predicted_volume_df.empty:
    predicted_volume_figure = plot_volume_dice_panels(
        predicted_volume_df.loc[
            predicted_volume_df["prediction_volume_status"].eq("ok")
        ],
        volume_dice_pairs,
        f"{SPLIT_LABELS[PLOT_SPLIT]}: volume predito × referência",
        y_column="predicted_volume_ml",
        volume_plot_xlim=volume_plot_xlim,
    )
if predicted_volume_figure is not None:
    predicted_volume_figure.show()

## 9. Erros e interpretação

O CSV preserva os erros para nova tentativa na execução seguinte. Os resultados descrevem a classe coronariana de referência, sem separar ramos esquerdo/direito e sem aplicar downsampling, threshold ou morfologia. Figuras e tabelas são exibidas aqui; apenas o CSV compacto é persistido.


In [ ]:
if error_df.empty:
    print("Nenhum erro registrado.")
else:
    display(error_df.loc[:, ["IMG_ID", "label_file", "error"]].reset_index(drop=True))